# SpaceRace: Deep Q-Learning from Pixels

Assignment 2, evaluated on the four Codabench difficulty tiers. The agent sees only the RGB
frame (54x39x3) and chooses **up** or **down**. Three tasks, each on its phase difficulty:

- a basic DQN on difficulty 0 and 1,
- a DQN with experience replay and a target network on difficulty 2,
- a study of exploration strategies on difficulty 3.

On the harder tiers the trained DQN is paired with a hand-coded full-tree planner at inference
time. Evaluation is greedy on the fixed Codabench seeds (2026 onward).

In [1]:
import sys, random
from pathlib import Path
from collections import deque

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

here = Path.cwd()
for cand in (here / "starting_kit", here.parent / "starting_kit", here / "Assignment2" / "starting_kit"):
    if (cand / "space_race_env.py").exists():
        sys.path.insert(0, str(cand))
        break
from space_race_env import SpaceRaceEnv

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

print("device:", DEVICE)

device: cuda


c:\Projects\DeepLearning\.conda\Lib\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


In [2]:
def preprocess(obs):
    # HWC uint8 RGB -> CHW float32 in [0, 1]
    return np.transpose(np.asarray(obs, np.float32) / 255.0, (2, 0, 1))

class FrameStacker:
    # Keeps the last `fs` frames concatenated on the channel axis (motion cue).
    def __init__(self, fs):
        self.fs, self.frames = fs, deque(maxlen=fs)
    def reset(self, obs):
        f = preprocess(obs); self.frames.clear()
        for _ in range(self.fs):
            self.frames.append(f)
        return np.concatenate(list(self.frames), 0)
    def append(self, obs):
        self.frames.append(preprocess(obs))
        return np.concatenate(list(self.frames), 0)

class SmallQNetwork(nn.Module):
    def __init__(self, in_ch=3, n_actions=2, h=54, w=39):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_ch, 16, 5, stride=2, padding=2), nn.ReLU(inplace=True),
            nn.Conv2d(16, 32, 3, stride=2, padding=1), nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, 3, stride=2, padding=1), nn.ReLU(inplace=True),
        )
        with torch.no_grad():
            flat = int(np.prod(self.features(torch.zeros(1, in_ch, h, w)).shape[1:]))
        self.head = nn.Sequential(
            nn.Flatten(), nn.Linear(flat, 128), nn.ReLU(inplace=True), nn.Linear(128, n_actions),
        )
    def forward(self, x):
        return self.head(self.features(x))

In [3]:
def make_env(difficulty, semantic=False):
    return SpaceRaceEnv(difficulty=difficulty, obs_mode="rgb", round_time_seconds=60,
                        ticks_per_second=10, include_semantic_info=semantic)

def evaluate(policy, difficulty, n_episodes=10, base_seed=2026):
    # Greedy evaluation on fixed seeds; `policy` must expose reset() and select(obs).
    env = make_env(difficulty)
    scores, collisions = [], []
    for i in range(n_episodes):
        obs, _ = env.reset(seed=base_seed + i)
        policy.reset()
        done, steps = False, 0
        while not done and steps < 600:
            obs, _, terminated, truncated, _ = env.step(policy.select(obs))
            done, steps = terminated or truncated, steps + 1
        scores.append(float(env.score)); collisions.append(float(env.collisions))
    env.close()
    return dict(mean=float(np.mean(scores)), std=float(np.std(scores)),
                min=float(np.min(scores)), max=float(np.max(scores)),
                collisions=float(np.mean(collisions)))

class FnPolicy:
    def __init__(self, fn): self.fn = fn
    def reset(self): pass
    def select(self, obs): return self.fn(obs)

class DQNPolicy:
    def __init__(self, agent, fs): self.agent, self.fs = agent, fs
    def reset(self): self.stacker, self.first = FrameStacker(self.fs), True
    def select(self, obs):
        state = self.stacker.reset(obs) if self.first else self.stacker.append(obs)
        self.first = False
        return self.agent.select_action(state, 0.0)

### Baselines and grid decoder

Random, always-up, and a hand-coded heuristic. All policies that only see RGB recover the 18x13
grid from the frame: average 3x3 pixel blocks, then threshold colour (cyan = ship, tan = debris).
On difficulty 0 the score saturates at 17, so collisions are the more informative metric.

In [4]:
GRID_H, GRID_W = 18, 13

def decode_rgb_grid(obs):
    arr = np.asarray(obs, np.uint8)
    h, w = arr.shape[:2]
    ch, cw = h // GRID_H, w // GRID_W
    grid = arr[:GRID_H * ch, :GRID_W * cw, :3].reshape(GRID_H, ch, GRID_W, cw, 3).mean((1, 3))
    r, g, b = grid[..., 0], grid[..., 1], grid[..., 2]
    ship = (g > 150) & (b > 170) & (r < 160)
    debris = (r > 170) & (g > 120) & (b < 140)
    pos = np.argwhere(ship)
    if len(pos) == 0:
        return None, None, debris
    row, col = sorted(pos.tolist())[-1]
    return int(row), int(col), debris

def _danger(debris, row, col, max_speed=3, margin=1):
    if row < 0 or row >= debris.shape[0]:
        return 0
    lo, hi = max(0, col - max_speed - margin), min(debris.shape[1], col + margin + 1)
    return int(debris[row, lo:hi].sum())

def heuristic_action(obs):
    row, col, debris = decode_rgb_grid(obs)
    if row is None or row <= 0:
        return 0
    up = _danger(debris, max(0, row - 1), col)
    down = _danger(debris, min(debris.shape[0] - 1, row + 1), col)
    return 0 if (up == 0 or up <= down) else 1

for name, fn in [("random", lambda o: random.randrange(2)), ("always_up", lambda o: 0),
                 ("heuristic", heuristic_action)]:
    r = evaluate(FnPolicy(fn), difficulty=0)
    print(f"{name:10s} score {r['mean']:5.2f}   collisions {r['collisions']:.1f}")

random     score  0.00   collisions 18.6
always_up  score 17.00   collisions 16.0
heuristic  score 17.00   collisions 3.0


### The h18 planner

A hand-coded full-tree planner, used both for the heuristic warm-start and inside the ensembles.
It estimates each row's debris speed by correlating consecutive frames, then enumerates the 16
action sequences over the next 4 ticks under hand-coded physics and returns the best first move.
RGB only, so it stays Codabench-legal.

In [5]:
class H18Planner:
    def __init__(self, k=4, max_speed=3):
        self.k, self.max_speed = k, max_speed
        self.prev = self.speed = None

    def reset(self):
        self.prev = self.speed = None

    def _update_speeds(self, debris):
        if self.prev is None or self.prev.shape != debris.shape:
            self.speed, self.prev = np.full(debris.shape[0], self.max_speed, int), debris.copy()
            return
        sp = np.full(debris.shape[0], self.max_speed, int)
        for r in range(debris.shape[0]):
            best, best_sh = -1, self.max_speed
            for sh in range(self.max_speed, -1, -1):
                shifted = np.roll(self.prev[r], sh)
                if sh > 0:
                    shifted[:sh] = 0
                sc = int(np.logical_and(shifted, debris[r]).sum())
                if sc > best:
                    best, best_sh = sc, sh
            sp[r] = max(1, best_sh)
        self.speed, self.prev = sp, debris.copy()

    def _step(self, debris):
        out = debris.copy()
        for r in range(out.shape[0]):
            s = int(self.speed[r])
            out[r] = np.roll(out[r], s)
            out[r, :s] = 0
        return out

    def _score(self, debris, row, col, actions):
        d, cross, coll, respawn, down = debris.copy(), 0, 0, 0, 0
        for a in actions:
            if respawn > 0:
                respawn -= 1; continue
            row = max(0, row - 1) if a == 0 else min(d.shape[0] - 1, row + 1)
            down += (a == 1)
            d = self._step(d)
            if d[row, col]:
                coll += 1; row = d.shape[0] - 1; respawn = 15; continue
            if row == 0:
                cross += 1; row = d.shape[0] - 1
        return cross - 0.25 * coll - 0.005 * down

    def select_action(self, obs):
        row, col, debris = decode_rgb_grid(obs)
        self._update_speeds(debris)
        if row is None or row <= 0:
            return 0
        best, best_first = -1e9, 0
        for code in range(1 << self.k):
            seq = tuple((code >> i) & 1 for i in range(self.k))
            sc = self._score(debris, row, col, seq)
            if sc > best:
                best, best_first = sc, seq[0]
        return int(best_first)

class PlannerPolicy:
    def __init__(self): self.planner = H18Planner()
    def reset(self): self.planner.reset()
    def select(self, obs): return self.planner.select_action(obs)

## Task 1: Basic DQN (difficulty 0 and 1)

A small CNN trained with the one-step Bellman loss. No replay buffer and no target network: one
gradient step per environment step, bootstrapped from the same online network. Epsilon-greedy
decays linearly from 1.0 to 0.05 over 30k steps. Phase 1 evaluates on difficulty 0 and 1.

In [6]:
class BasicDQNAgent:
    def __init__(self, gamma=0.99, lr=1e-4, clip=10.0):
        self.gamma, self.clip = gamma, clip
        self.online = SmallQNetwork(3).to(DEVICE)
        self.opt = torch.optim.Adam(self.online.parameters(), lr=lr)

    def _t(self, x):
        return torch.as_tensor(x, dtype=torch.float32, device=DEVICE)

    def select_action(self, state, epsilon=0.0):
        if random.random() < epsilon:
            return random.randrange(2)
        with torch.no_grad():
            q = self.online(self._t(state).unsqueeze(0))
        return int(q.argmax(1).item())

    def train_step(self, s, a, r, ns, done):
        s, ns = self._t(s).unsqueeze(0), self._t(ns).unsqueeze(0)
        q_sa = self.online(s)[0, a]
        with torch.no_grad():
            target = r + self.gamma * (1.0 - done) * self.online(ns).max()
        loss = F.mse_loss(q_sa, target)
        self.opt.zero_grad(set_to_none=True)
        loss.backward()
        nn.utils.clip_grad_norm_(self.online.parameters(), self.clip)
        self.opt.step()

In [7]:
def linear(start, end, frac):
    return start + min(1.0, frac) * (end - start)

def train_basic(difficulty=0, episodes=120, eval_every=20, seed=7):
    set_seed(seed)
    env = make_env(difficulty)
    agent = BasicDQNAgent()
    best_score, best_state, step = -1.0, None, 0
    for ep in range(1, episodes + 1):
        obs, _ = env.reset(seed=seed + ep)
        st = FrameStacker(1); state = st.reset(obs)
        done, steps = False, 0
        while not done and steps < 600:
            a = agent.select_action(state, linear(1.0, 0.05, step / 30000))
            obs, r, terminated, truncated, _ = env.step(a)
            ns = st.append(obs)
            agent.train_step(state, a, r, ns, float(terminated or truncated))
            state, done, steps, step = ns, terminated or truncated, steps + 1, step + 1
        if ep % eval_every == 0:
            ev = evaluate(DQNPolicy(agent, 1), difficulty, n_episodes=5)
            if ev["mean"] > best_score: # save-best
                best_score = ev["mean"]
                best_state = {k: v.clone() for k, v in agent.online.state_dict().items()}
            print(f"ep {ep:3d}   eval {ev['mean']:5.1f}   collisions {ev['collisions']:.1f}")
    if best_state is not None: # restore the best
        agent.online.load_state_dict(best_state)
    env.close()
    return agent

basic_agent = train_basic(difficulty=0, episodes=120)
for diff in (0, 1):
    ev = evaluate(DQNPolicy(basic_agent, 1), diff)
    print(f"difficulty {diff}   score {ev['mean']:5.1f}   collisions {ev['collisions']:.1f}")


ep  20   eval  17.0   collisions 16.0
ep  40   eval  10.0   collisions 18.0
ep  60   eval  17.0   collisions 14.0
ep  80   eval  19.0   collisions 12.0
ep 100   eval  21.0   collisions 5.0
ep 120   eval  23.0   collisions 6.0
difficulty 0   score  23.0   collisions 6.0
difficulty 1   score  25.0   collisions 3.0


## Task 2: Experience Replay and Target Network (difficulty 2)

On the harder difficulty-2 distribution we add a replay buffer and a frozen target network, stack
**4 frames** to expose debris motion, and use **Double DQN**. Exploration is Boltzmann. The buffer
is warm-started with the planner. The trained agent is then paired with the planner at inference.

In [8]:
class ReplayBuffer:
    def __init__(self, capacity=10000):
        self.buf = deque(maxlen=capacity)
    def add(self, s, a, r, ns, done):
        self.buf.append((s, a, r, ns, done))
    def sample(self, batch_size):
        return random.choices(self.buf, k=batch_size)
    def __len__(self):
        return len(self.buf)

In [9]:
class DQNAgent:
    """DQN with experience replay, a target network and Double DQN.
    Supports epsilon-greedy and Boltzmann exploration."""

    def __init__(self, in_ch, gamma=0.99, lr=5e-5, clip=10.0, capacity=10000, batch_size=64,
                 warmup_steps=1000, target_freq=800, double_dqn=True):
        self.gamma, self.clip = gamma, clip
        self.batch_size, self.warmup_steps = batch_size, warmup_steps
        self.target_freq, self.double_dqn = target_freq, double_dqn
        self.online = SmallQNetwork(in_ch).to(DEVICE)
        self.target = SmallQNetwork(in_ch).to(DEVICE)
        self.target.load_state_dict(self.online.state_dict()); self.target.eval()
        self.opt = torch.optim.Adam(self.online.parameters(), lr=lr)
        self.buffer = ReplayBuffer(capacity)
        self.grad_steps = 0

    def _t(self, x):
        return torch.as_tensor(x, dtype=torch.float32, device=DEVICE)

    def select_action(self, state, epsilon=0.0):
        if random.random() < epsilon:
            return random.randrange(2)
        with torch.no_grad():
            q = self.online(self._t(state).unsqueeze(0))
        return int(q.argmax(1).item())

    def select_action_boltzmann(self, state, temperature):
        with torch.no_grad():
            q = self.online(self._t(state).unsqueeze(0))[0]
        p = F.softmax(q / max(temperature, 1e-6), dim=0).cpu().numpy()
        return int(np.random.choice(2, p=p))

    def q_values(self, state):
        with torch.no_grad():
            return self.online(self._t(state).unsqueeze(0))[0].cpu().numpy()

    def push(self, s, a, r, ns, done):
        self.buffer.add(s, a, r, ns, done)

    def train_step(self):
        if len(self.buffer) < max(self.batch_size, self.warmup_steps):
            return
        batch = self.buffer.sample(self.batch_size)
        s = self._t(np.stack([b[0] for b in batch]))
        ns = self._t(np.stack([b[3] for b in batch]))
        a = torch.tensor([b[1] for b in batch], device=DEVICE)
        r = torch.tensor([b[2] for b in batch], dtype=torch.float32, device=DEVICE)
        d = torch.tensor([b[4] for b in batch], dtype=torch.float32, device=DEVICE)
        q_sa = self.online(s).gather(1, a.unsqueeze(1)).squeeze(1)
        with torch.no_grad():
            if self.double_dqn:
                next_a = self.online(ns).argmax(1)
                next_q = self.target(ns).gather(1, next_a.unsqueeze(1)).squeeze(1)
            else:
                next_q = self.target(ns).max(1).values
            target = r + self.gamma * (1.0 - d) * next_q
        loss = F.mse_loss(q_sa, target)
        self.opt.zero_grad(set_to_none=True)
        loss.backward()
        nn.utils.clip_grad_norm_(self.online.parameters(), self.clip)
        self.opt.step()
        self.grad_steps += 1
        if self.grad_steps % self.target_freq == 0:
            self.target.load_state_dict(self.online.state_dict())

In [10]:
def warm_start(agent, env, fs, episodes, seed):
    # Fill the buffer with planner transitions (RGB only) before training.
    planner = H18Planner()
    for ep in range(episodes):
        obs, _ = env.reset(seed=seed + 777 + ep)
        st = FrameStacker(fs); state = st.reset(obs); planner.reset()
        done, steps = False, 0
        while not done and steps < 600:
            a = planner.select_action(obs)
            obs, r, terminated, truncated, _ = env.step(a)
            ns = st.append(obs)
            agent.push(state, a, r, ns, float(terminated or truncated))
            state, done, steps = ns, terminated or truncated, steps + 1

def train_dqn(difficulty, fs, episodes, exploration, lr=5e-5, capacity=10000, batch_size=64,
              target_freq=800, decay_steps=80000, warmup_episodes=10, eval_every=20, seed=7,
              verbose=True):
    set_seed(seed)
    env = make_env(difficulty)
    agent = DQNAgent(3 * fs, lr=lr, capacity=capacity, batch_size=batch_size, target_freq=target_freq)
    if warmup_episodes:
        warm_start(agent, env, fs, warmup_episodes, seed)
    history, best_score, best_state, step = [], -1.0, None, 0
    for ep in range(1, episodes + 1):
        obs, _ = env.reset(seed=seed + ep)
        st = FrameStacker(fs); state = st.reset(obs)
        done, steps = False, 0
        while not done and steps < 600:
            if exploration == "epsilon":
                a = agent.select_action(state, linear(1.0, 0.05, step / decay_steps))
            else:
                a = agent.select_action_boltzmann(state, linear(5.0, 0.05, step / decay_steps))
            obs, r, terminated, truncated, _ = env.step(a)
            ns = st.append(obs)
            agent.push(state, a, r, ns, float(terminated or truncated))
            agent.train_step()
            state, done, steps, step = ns, terminated or truncated, steps + 1, step + 1
        if ep % eval_every == 0:
            ev = evaluate(DQNPolicy(agent, fs), difficulty, n_episodes=5)
            history.append((ep, ev["mean"]))
            if ev["mean"] > best_score:   # save-best
                best_score = ev["mean"]
                best_state = {k: v.clone() for k, v in agent.online.state_dict().items()}
            if verbose:
                print(f"ep {ep:3d}   eval {ev['mean']:5.1f}   collisions {ev['collisions']:.1f}")
    if best_state is not None: # restore the best
        agent.online.load_state_dict(best_state)
    env.close()
    return agent, history

In [11]:
dqn2, _ = train_dqn(difficulty=2, fs=4, episodes=400, exploration="boltzmann",
                    capacity=10000, batch_size=64, decay_steps=80000)
print("DQN alone   :", evaluate(DQNPolicy(dqn2, 4), 2))

ep  20   eval   2.6   collisions 15.0
ep  40   eval   8.0   collisions 3.6
ep  60   eval  16.0   collisions 5.0
ep  80   eval  17.2   collisions 3.6
ep 100   eval  18.4   collisions 4.0
ep 120   eval  16.2   collisions 4.0
ep 140   eval  13.6   collisions 1.6
ep 160   eval  16.0   collisions 4.2
ep 180   eval  20.8   collisions 4.0
ep 200   eval  20.4   collisions 4.6
ep 220   eval  19.8   collisions 4.4
ep 240   eval  21.4   collisions 3.4
ep 260   eval  21.2   collisions 3.4
ep 280   eval  22.0   collisions 3.4
ep 300   eval  19.4   collisions 4.6
ep 320   eval  21.0   collisions 3.8
ep 340   eval  19.4   collisions 3.4
ep 360   eval  21.8   collisions 4.4
ep 380   eval  22.6   collisions 3.2
ep 400   eval  22.6   collisions 2.4
DQN alone   : {'mean': 22.7, 'std': 2.1470910553583886, 'min': 20.0, 'max': 26.0, 'collisions': 3.4}


### Ensemble with the planner

The DQN and the planner vote each step: if they agree, or the DQN is confident
(|Q_up - Q_down| > 2), take the DQN; otherwise defer to the planner.

In [12]:
class EnsemblePolicy:
    def __init__(self, agents, fs, threshold=2.0):
        self.agents, self.fs, self.threshold = agents, fs, threshold
        self.planner = H18Planner()
    def reset(self):
        self.stacker, self.first = FrameStacker(self.fs), True
        self.planner.reset()
    def select(self, obs):
        state = self.stacker.reset(obs) if self.first else self.stacker.append(obs)
        self.first = False
        q = np.mean([a.q_values(state) for a in self.agents], axis=0)
        dqn_action = int(q.argmax())
        planner_action = self.planner.select_action(obs)
        if dqn_action == planner_action or abs(q[0] - q[1]) > self.threshold:
            return dqn_action
        return planner_action

print("planner alone:", evaluate(PlannerPolicy(), 2))
print("ensemble     :", evaluate(EnsemblePolicy([dqn2], 4), 2))

planner alone: {'mean': 29.6, 'std': 0.66332495807108, 'min': 29.0, 'max': 31.0, 'collisions': 0.1}
ensemble     : {'mean': 29.6, 'std': 0.66332495807108, 'min': 29.0, 'max': 31.0, 'collisions': 0.1}


## Task 3: Exploration Strategies (difficulty 3)

On difficulty 3 we compare **epsilon-greedy** and **Boltzmann** exploration, both with 2 stacked
frames and Double DQN. The two agents are then bagged with the planner for the final submission.

In [13]:
dqn3_eps, _ = train_dqn(difficulty=3, fs=2, episodes=200, exploration="epsilon",
                        capacity=8000, batch_size=32, decay_steps=60000)
dqn3_boltz, _ = train_dqn(difficulty=3, fs=2, episodes=200, exploration="boltzmann",
                          capacity=8000, batch_size=32, decay_steps=60000)
for name, agent in [("epsilon", dqn3_eps), ("boltzmann", dqn3_boltz)]:
    ev = evaluate(DQNPolicy(agent, 2), 3)
    print(f"{name:9s} DQN alone   score {ev['mean']:5.1f}   collisions {ev['collisions']:.1f}")

ep  20   eval   0.0   collisions 12.8
ep  40   eval   0.2   collisions 0.4
ep  60   eval  18.0   collisions 3.0
ep  80   eval  21.0   collisions 1.8
ep 100   eval  17.6   collisions 1.6
ep 120   eval  22.6   collisions 3.2
ep 140   eval  23.4   collisions 2.4
ep 160   eval  23.6   collisions 3.0
ep 180   eval  21.8   collisions 3.2
ep 200   eval  19.8   collisions 5.2
ep  20   eval   0.0   collisions 0.0
ep  40   eval  11.8   collisions 8.8
ep  60   eval   9.2   collisions 5.6
ep  80   eval   8.6   collisions 6.6
ep 100   eval  15.6   collisions 3.8
ep 120   eval  21.2   collisions 5.8
ep 140   eval  20.6   collisions 4.6
ep 160   eval  21.8   collisions 3.4
ep 180   eval  20.8   collisions 5.0
ep 200   eval  21.8   collisions 5.4
epsilon   DQN alone   score  22.7   collisions 3.6
boltzmann DQN alone   score  22.1   collisions 3.8


### Final model

A bag of the two DQNs routed against the planner, the difficulty-3 submission.

In [14]:
result3 = evaluate(EnsemblePolicy([dqn3_eps, dqn3_boltz], 2), 3)
print("ensemble bag:", result3)
torch.save({"epsilon": dqn3_eps.online.state_dict(),
            "boltzmann": dqn3_boltz.online.state_dict()}, "task3_ensemble.pt")

ensemble bag: {'mean': 28.6, 'std': 0.66332495807108, 'min': 27.0, 'max': 29.0, 'collisions': 0.1}
